## Connecting to stacks
Make sure to run docker for timescaledb and that you populated the needed tokens in the .env! 

You can run this code either locally in this notebook (as long as you pip install all the dependencies in the venv) or on the jupyter notebook in the local host after running the docker containers

### Saved db
When you run this code, it will generate two databases in the docker container (within timescaledb):
- jkp_characteristics (raw characteristics)
- jkp_characteristics_with_sector (with industry mappings, to be cleaned in... )

In [ ]:
import os
from sqlalchemy import create_engine, text

def is_docker_environment() -> bool:
    """
    Detect if running inside a Docker/Jupyter container.
    """
    # 1) Standard Docker marker
    if os.path.exists("/.dockerenv"):
        return True

    # 2) Heuristic: Jupyter in container often sets this,
    #    and our compose uses PGHOST=timescaledb inside the container.
    if os.environ.get("JUPYTER_SERVER_ROOT"):
        return True

    return False

def get_default_pg_host() -> str:
    """
    Choose sensible default host based on environment:
    - Inside Docker/Jupyter container: talk to 'timescaledb' service
    - Local (Mac): talk to Docker via 'localhost'
    """
    return "timescaledb" if is_docker_environment() else "localhost"

# Respect explicit PGHOST if user sets it; otherwise use env-aware default
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD", "postgres")
PGHOST = os.getenv("PGHOST", get_default_pg_host())
PGPORT = os.getenv("PGPORT", "5432")
PGDATABASE = os.getenv("PGDATABASE", "factor_db")

db_url = f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}"
engine = create_engine(db_url, future=True)

try:
    with engine.begin() as conn:
        conn.execute(text("SELECT 1;"))
    env = "Docker/Jupyter" if is_docker_environment() else "Local"
    print(f"✅ Connected to TimescaleDB/Postgres ({env})")
    print(f"   URL: {db_url}")
except Exception as e:
    print(f"❌ Failed to connect to TimescaleDB/Postgres: {e}")
    raise

In [ ]:
import os
import wrds
import pandas as pd
import getpass  # kept in case you later want manual mode

print("🔐 Establishing WRDS connection...")

def get_wrds_connection():
    """
    Use WRDS_USERNAME / WRDS_PASSWORD from environment if present.
    Do NOT fall back to interactive input when they exist.
    """
    username = os.environ.get("WRDS_USERNAME")
    password = os.environ.get("WRDS_PASSWORD")

    print("DEBUG WRDS_USERNAME:", username)

    if username and password:
        print(f"Using WRDS credentials from environment for user: {username}")
        try:
            conn = wrds.Connection(wrds_username=username, wrds_password=password)
            print("🔗 WRDS connection established successfully")
            return conn
        except Exception as e:
            # Show the real reason and stop here
            print(f"❌ Failed to connect with environment credentials: {e}")
            raise
    else:
        # Only if you REALLY want interactive when env vars are missing
        print("⚠️  No WRDS credentials found in environment variables")
        print("💡 Add WRDS_USERNAME and WRDS_PASSWORD to your .env for automatic login")


# Establish WRDS connection
wrds_db = get_wrds_connection()

## Loading WRDS

### Checking Date Range

In [ ]:
date_range = wrds_db.raw_sql("""
    SELECT
        MIN(eom) AS min_eom,
        MAX(eom) AS max_eom
    FROM contrib.global_factor;
""")

print(date_range)

In [ ]:
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import pandas as pd

class JKPDataLoader:
    """
    Modular data loader for JKP (Jensen, Kelly, Pedersen) global characteristics data
    Validated against JKP_Characteristics_README.md and characteristics.csv
    """
    
    # Factor mappings based on JKP documentation and academic validation
    FACTOR_MODELS = {
        'fama_french_5': {
            # Core Fama-French 5 factors - validated against JKP documentation
            'value': ['be_me', 'ni_me', 'ocf_me'],  # Book-to-market (primary), earnings yield, cash flow yield
            'size': ['market_equity'],  # Market equity (primary FF size measure)
            'momentum': ['ret_12_1'],  # 12-1 month momentum (Jegadeesh Titman 1993)
            'profitability': ['gp_at', 'ni_at'],  # Gross profits/assets (Novy-Marx 2013), ROA
            'investment': ['at_gr1', 'oaccruals_at'],  # Asset growth (Cooper et al 2008), Operating accruals (Sloan 1996)
            'market': ['ret_exc', 'beta_60m']  # Market excess returns and beta
        },
        'fama_french_5_extended': {
            # Extended version with more characteristics per factor
            'value': ['be_me', 'ni_me', 'ocf_me', 'sale_me', 'fcf_me'],
            'size': ['market_equity', 'me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'ret_12_7'],
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'market': ['ret_exc', 'beta_60m', 'beta_252d']
        },
        'quality_comprehensive': {
            # Comprehensive quality-focused model
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be', 'ocf_at'],
            'quality_composite': ['f_score', 'qmj', 'qmj_prof', 'qmj_safety'],
            'earnings_quality': ['ni_ar1', 'earnings_variability', 'oaccruals_at'],
            'growth_quality': ['sale_gr1', 'at_gr1', 'qmj_growth'],
            'financial_strength': ['at_be', 'cash_at', 'z_score']
        },
        'trading_frictions': {
            # Trading frictions and liquidity factors
            'size': ['market_equity', 'me'],
            'volatility': ['ivol_capm_252d', 'ivol_ff3_21d', 'rvol_21d'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'beta': ['beta_60m', 'beta_252d', 'betadown_252d'],
            'skewness': ['iskew_capm_21d', 'rskew_21d', 'coskew_21d'],
            'extremes': ['rmax1_21d', 'rmax5_21d', 'rmax5_rvol_21d']
        },
        'academic_research': {
            # Comprehensive academic research model
            'value': ['be_me', 'ni_me', 'fcf_me', 'sale_me'],
            'momentum': ['ret_12_1', 'ret_6_1', 'resff3_12_1'],
            'profitability': ['gp_at', 'ni_at', 'ope_be', 'f_score'],
            'investment': ['at_gr1', 'capx_gr1', 'oaccruals_at', 'netis_at'],
            'volatility': ['ivol_capm_252d', 'beta_60m'],
            'liquidity': ['ami_126d', 'turnover_126d', 'zero_trades_126d'],
            'quality': ['qmj', 'qmj_prof', 'qmj_safety'],
            'intangibles': ['rd_me', 'age', 'tangibility']
        },
        'basic': {
            # Simple 4-factor model
            'returns': ['ret_exc'],
            'size': ['market_equity'],
            'value': ['be_me'],
            'momentum': ['ret_12_1']
        }
    }
    
    def __init__(self, wrds_connection):
        self.wrds_db = wrds_connection
        
    def get_available_characteristics(self) -> List[str]:
        """Get all available characteristics from the JKP dataset"""
        schema = self.wrds_db.get_table(library="contrib", table="global_factor", obs=0)
        return schema.columns.tolist()
    
    def validate_characteristics(self, characteristics: List[str]) -> List[str]:
        """Validate that requested characteristics exist in the dataset"""
        available_chars = self.get_available_characteristics()
        valid_chars = []
        invalid_chars = []
        
        for char in characteristics:
            if char in available_chars:
                valid_chars.append(char)
            else:
                invalid_chars.append(char)
        
        if invalid_chars:
            print(f"Warning: Invalid characteristics removed: {invalid_chars}")
            print(f"Available similar characteristics: {[c for c in available_chars if any(inv in c for inv in invalid_chars)]}")
        
        return valid_chars
    
    def load_characteristics_data(self, 
                                factor_model: str = 'fama_french_5',
                                custom_factors: Optional[List[str]] = None,
                                months_back: int = 500,
                                countries: List[str] = ['USA'],
                                filters: Optional[Dict] = None) -> pd.DataFrame:
        """
        Load characteristics data with flexible parameters
        
        Parameters:
        -----------
        factor_model : str
            Predefined factor model ('fama_french_5', 'fama_french_5_extended', 'quality_comprehensive', 
            'trading_frictions', 'academic_research', 'basic')
        custom_factors : List[str], optional
            Custom list of factor names to load
        months_back : int
            Number of months of historical data to load
        countries : List[str]
            List of countries to include
        filters : Dict, optional
            Additional filters for data quality
        """
        
        # Determine which characteristics to load
        if custom_factors:
            characteristics = custom_factors
        elif factor_model in self.FACTOR_MODELS:
            # Flatten all factors in the model
            characteristics = []
            for factor_group in self.FACTOR_MODELS[factor_model].values():
                characteristics.extend(factor_group)
            # Remove duplicates
            characteristics = list(set(characteristics))
        else:
            raise ValueError(f"Unknown factor model: {factor_model}. Available models: {list(self.FACTOR_MODELS.keys())}")
        
        # Validate characteristics exist in dataset
        characteristics = self.validate_characteristics(characteristics)
        
        if not characteristics:
            raise ValueError("No valid characteristics found!")
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months_back * 30)  # Approximate
        
        # Default filters for data quality (based on JKP recommendations)
        default_filters = {
            'common': 1,        # Common stocks only
            'exch_main': 1,     # Main exchange only
            'primary_sec': 1,   # Primary security only
            'obs_main': 1       # Main observation only
        }
        
        if filters:
            default_filters.update(filters)
        
        # Build the SQL query
        filter_conditions = []
        for key, value in default_filters.items():
            filter_conditions.append(f"{key}={value}")
        
        # Add country filter
        if countries:
            country_list = "', '".join(countries)
            filter_conditions.append(f"excntry IN ('{country_list}')")
        
        # Add date filter
        filter_conditions.append(f"eom >= '{start_date.strftime('%Y-%m-%d')}'")
        
        # Build characteristics string
        char_string = ', '.join(characteristics)
        
        sql_query = f"""
        SELECT id, eom, gvkey, permno, size_grp, {char_string}
        FROM contrib.global_factor
        WHERE {' AND '.join(filter_conditions)}
        ORDER BY eom DESC
        """
        
        print(f"Loading {len(characteristics)} characteristics for last {months_back} months...")
        print(f"Factor model: {factor_model}")
        print(f"Characteristics: {characteristics}")
        
        return self.wrds_db.raw_sql(sql_query)
    
    def get_factor_model_info(self, model_name: str) -> Dict:
        """Get information about a specific factor model"""
        if model_name in self.FACTOR_MODELS:
            return self.FACTOR_MODELS[model_name]
        else:
            return {}
    
    def list_available_models(self) -> List[str]:
        """List all available factor models"""
        return list(self.FACTOR_MODELS.keys())
    
    def print_model_summary(self, model_name: str):
        """Print detailed summary of a factor model"""
        if model_name not in self.FACTOR_MODELS:
            print(f"Model '{model_name}' not found. Available models: {self.list_available_models()}")
            return
        
        model = self.FACTOR_MODELS[model_name]
        print(f"\n{model_name.upper()} FACTOR MODEL")
        print("=" * 50)
        
        total_chars = 0
        for factor_group, characteristics in model.items():
            print(f"\n{factor_group.upper()}:")
            for char in characteristics:
                print(f"  • {char}")
            total_chars += len(characteristics)
        
        print(f"\nTotal characteristics: {total_chars}")
        print(f"Factor groups: {len(model)}")

# Initialize the data loader
data_loader = JKPDataLoader(wrds_db)

print("📊 JKP Data Loader Initialized")
print("=" * 40)

print("\nAvailable factor models:")
for model in data_loader.list_available_models():
    print(f"  📈 {model}")

# Print detailed summary of the Fama-French 5 model
data_loader.print_model_summary('fama_french_5')

# Load validated Fama-French 5-factor model data
print(f"\n🔄 Loading data...")
usa_data = data_loader.load_characteristics_data(
    factor_model='fama_french_5',
    months_back=12,
    countries=['USA']
)

print(f"\n✅ Successfully loaded {len(usa_data)} observations")
print(f"📅 Date range: {usa_data['eom'].min()} to {usa_data['eom'].max()}")
print(f"🏢 Unique companies: {usa_data['gvkey'].nunique()}")
print(f"📊 Data shape: {usa_data.shape}")

usa_data.head()

### Fama-French 5 Factor Model Characteristics

#### Value Factor Characteristics
- **be_me**: Book equity to market equity ratio (classic book-to-market)
- **at_me**: Total assets to market equity ratio
- **sale_me**: Sales to market equity ratio
- **ni_me**: Net income to market equity ratio  
- **ocf_me**: Operating cash flow to market equity ratio
- **ebitda_me**: EBITDA to market equity ratio

#### CAPM/Market Factor Characteristics  
- **ret_exc**: Excess returns (return minus risk-free rate)
- **beta_60m**: 60-month market beta
- **beta_252d**: 252-day (1-year) market beta

#### Size Factor Characteristics
- **me**: Market equity (shares outstanding × price)
- **market_equity**: Total market value of equity
- **sale_me**: Sales-to-market equity (revenue efficiency)

#### Momentum Factor Characteristics
- **ret_12_1**: 12-month return skipping most recent month
- **ret_6_1**: 6-month return skipping most recent month  
- **ret_12_7**: 12-month return skipping most recent 7 months
- **ret_9_1**: 9-month return skipping most recent month

#### Quality/Profitability Factor Characteristics
- **ni_at**: Net income to total assets (ROA)
- **ni_be**: Net income to book equity (ROE)
- **roe_be_std**: ROE standard deviation (earnings stability)
- **ocf_at**: Operating cash flow to total assets
- **gp_at**: Gross profits to total assets
- **f_score**: Piotroski F-Score (composite quality measure)

#### Additional Factor Models Available
- **profitability**: Focus on earnings, cash flow, margins, returns
- **investment**: Asset growth, capex, financing measures  
- **risk**: Volatility, beta, skewness, maximum return measures
- **basic**: Simple 4-factor model (returns, size, value, momentum)

## Writing onto TimescaleDB

In [ ]:
usa_data.to_sql(
    "jkp_characteristics",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=10_000,
)

### Testing query

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    df = pd.read_sql(
        text("SELECT * FROM jkp_characteristics LIMIT 100;"),
        conn,
    )

df.head()

## Adding Sector/industry mappings and Tickers

In [ ]:
gvkey_permno = wrds_db.raw_sql("""
    SELECT gvkey, lpermno as permno, lpermco as permco
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU', 'LC') -- Active links
""")

crsp_tickers = wrds_db.raw_sql("""
    SELECT permno, ticker
    FROM crsp.stocknames
""")

comp_names = wrds_db.raw_sql("""
    SELECT gvkey, conm
    FROM comp.company
""")

# Merge the tables to get GVKEY → PERMNO → Ticker
gvkey_to_ticker = gvkey_permno.merge(crsp_tickers, on="permno", how="left")

gvkey_to_ticker = gvkey_to_ticker.merge(comp_names, on="gvkey", how="left")

gvkey_to_ticker.drop_duplicates(inplace=True)

gvkey_to_ticker.head()

In [ ]:
missing = gvkey_to_ticker[gvkey_to_ticker['conm'].isna()]
missing

In [ ]:
gvkey_to_ticker[gvkey_to_ticker['ticker'] == 'BCS']

In [ ]:
gvkey_to_ticker.dtypes

In [ ]:
empty = gvkey_to_ticker[gvkey_to_ticker['ticker'].isna()]

In [ ]:
empty

In [ ]:
test = gvkey_to_ticker[gvkey_to_ticker['ticker'] == 'UNH']
test

In [ ]:
gics_kaggle_url = (
    "https://storage.googleapis.com/kagglesdsdata/datasets/1097064/8252091/"
    "gics-map-2023.csv?X-Goog-Algorithm=GOOG4-RSA-SHA256&X-Goog-Credential="
    "gcp-kaggle-com%40kaggle-161607.iam.gserviceaccount.com%2F20251205%2Fauto"
    "%2Fstorage%2Fgoog4_request&X-Goog-Date=20251205T192523Z&X-Goog-Expires="
    "259200&X-Goog-SignedHeaders=host&X-Goog-Signature="
    "c329197b388bb9228870e3826b9592fd9256a5a0bb52158e1c633b7ed3453fc594b4be1"
    "dbbd3b6973c201506407f410512d4546a351a1d1cdecb202ddaef906207bf5d376b9794"
    "7d19893bf398ca8d22a566887483be673dccdd8748f8d8c95d36131a0654bd114d2c3cc"
    "db9deb4b57e01666e2db8bf418cc241a52073fd111f0764643fb88a98197f6c5e41f742"
    "1452424f69571b7972f8f5337de7039c9f977e94f39b52e60b9781c11405809d429878a"
    "0dfaea19cd69b384e6366bd1932ca2996a61c5aca9e14005e3cadf3f40a68b606ff35de"
    "9fb9ad4c942a53dfbbebdf3edd026b1fe88dc7964ad3a33b93eeddbd54d52a9a3aaeb78"
    "cb441c0a7e62e04"
)

gics_map_df = pd.read_csv(gics_kaggle_url)

In [ ]:
gics_map_df.columns

In [ ]:
# === 1. Get latest GICS industry/sector per gvkey ===
gics_raw = wrds_db.raw_sql("""
    SELECT
        gvkey,
        gsector,          -- 2‑digit GICS sector
        ggroup,           -- 4‑digit GICS industry group
        gind,             -- 6‑digit GICS industry
        gsubind           -- 8‑digit GICS sub‑industry
    FROM comp.company
    WHERE gsector IS NOT NULL
""")

# keep most recent classification per gvkey
gics_latest = gics_raw.drop_duplicates(subset=["gvkey"])

print("GICS rows (unique gvkeys):", len(gics_latest))


print("gvkey_to_ticker rows:", len(gvkey_to_ticker))

# === 1b. Adding name mappings (sector) ===

def _norm_code(series):
    # e.g. 10.0 -> "10", 3510.0 -> "3510"
    s = series.astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    return s

# normalize WRDS codes
gics_latest["gsector_str"]  = _norm_code(gics_latest["gsector"])
gics_latest["ggroup_str"]   = _norm_code(gics_latest["ggroup"])
gics_latest["gind_str"]     = _norm_code(gics_latest["gind"])
gics_latest["gsubind_str"]  = _norm_code(gics_latest["gsubind"])

# normalize Kaggle codes
gics_map_df["SectorId_str"]         = _norm_code(gics_map_df["SectorId"])
gics_map_df["IndustryGroupId_str"]  = _norm_code(gics_map_df["IndustryGroupId"])
gics_map_df["IndustryId_str"]       = _norm_code(gics_map_df["IndustryId"])
gics_map_df["SubIndustryId_str"]    = _norm_code(gics_map_df["SubIndustryId"])

# keep only what we need from Kaggle
gics_map_small = gics_map_df[
    [
        "SectorId_str",
        "IndustryGroupId_str",
        "IndustryId_str",
        "SubIndustryId_str",
        "Sector",
        "IndustryGroup",
        "Industry",
        "SubIndustry",
        "SubIndustryDescription",
    ]
].drop_duplicates()

gics_with_names = gics_latest.merge(
    gics_map_small,
    left_on=["gsector_str", "ggroup_str", "gind_str", "gsubind_str"],
    right_on=["SectorId_str", "IndustryGroupId_str", "IndustryId_str", "SubIndustryId_str"],
    how="left",
)

# drop helper columns if you want
gics_with_names = gics_with_names.drop(
    columns=[
        "gsector_str",
        "ggroup_str",
        "gind_str",
        "gsubind_str",
        "SectorId_str",
        "IndustryGroupId_str",
        "IndustryId_str",
        "SubIndustryId_str",
    ]
)

# === 2. Merge GICS onto gvkey_to_ticker (gvkey‑level industry mapping) ===
gvkey_with_sector = gvkey_to_ticker.merge(
    gics_with_names,
    on="gvkey",
    how="left",
)

gvkey_with_sector.head()


In [ ]:
test = gvkey_with_sector[gvkey_with_sector['ticker'] == 'UNH']
test

### GICS / Identifier Columns Explained

This section documents the main identifier and industry‐classification columns we merged:

| Column   | Source           | Meaning                                                                 |
|----------|------------------|-------------------------------------------------------------------------|
| `gvkey`  | COMPUSTAT (comp) | Permanent COMPUSTAT company identifier (firm‑level, survives ticker changes). |
| `permno` | CRSP (crsp)      | Permanent CRSP security identifier (share‑class / listing level).      |
| `permco` | CRSP (crsp)      | Permanent CRSP company identifier (issuer level, may map to many `permno`). |
| `ticker` | CRSP (crsp)      | Exchange ticker symbol (can change over time, not stable ID).          |
| `conm`   | COMPUSTAT (comp) | Company name from COMPUSTAT.                                           |
| `gsector`| COMPUSTAT (comp) | 2‑digit **GICS sector** code.                                          |
| `ggroup` | COMPUSTAT (comp) | 4‑digit **GICS industry group** code.                                  |
| `gind`   | COMPUSTAT (comp) | 6‑digit **GICS industry** code.                                        |
| `gsubind`| COMPUSTAT (comp) | 8‑digit **GICS sub‑industry** code (most granular level).              |

Example row for UNH:

| gvkey  | permno | permco | ticker | conm                     | gsector | ggroup | gind   | gsubind  |
|--------|--------|--------|--------|--------------------------|---------|--------|--------|----------|
| 010903 | 92655  | 7267   | UNH    | UNITEDHEALTH GROUP INC   | 35      | 3510   | 351020 | 35102030 |

#### GICS Sector Codes (WRDS / COMPUSTAT)

The `gsector` column is a 2‑digit numeric sector code following the Global Industry Classification Standard (GICS). Common sector codes include (exact availability depends on WRDS / S&P version):

- `10` – Energy  
- `15` – Materials  
- `20` – Industrials  
- `25` – Consumer Discretionary  
- `30` – Consumer Staples  
- `35` – Health Care  
- `40` – Financials  
- `45` – Information Technology  
- `50` – Communication Services  
- `55` – Utilities  
- `60` – Real Estate 

## Joining Sector with charactersitics data + Cleaning (multiple tickers for the same gvkeys)

In [ ]:
from sqlalchemy import text

# 1) Load characteristics from TimescaleDB
with engine.connect() as conn:
    chars_tsdb = pd.read_sql(
        text("SELECT * FROM jkp_characteristics;"),
        conn,
)

print("Rows from TimescaleDB jkp_characteristics:", len(chars_tsdb))
print("Unique gvkeys in TSDB:", chars_tsdb["gvkey"].nunique())

# 2) Join with gvkey_with_sector (which already has ticker + GICS names)
chars_with_sector = chars_tsdb.merge(
    gvkey_with_sector,
    on="gvkey",
    how="left",   # keep all TSDB rows; use 'inner' if you only want matched gvkeys
)

print("Rows in chars_with_sector:", len(chars_with_sector))
print("Columns in chars_with_sector:", chars_with_sector.shape[1])
chars_with_sector.head()

In [ ]:
#initial loading: 14813
chars_with_sector.drop_duplicates(inplace=True)
chars_with_sector.shape

In [ ]:
dupe_mask = chars_with_sector.duplicated(subset=["gvkey", "eom"], keep=False)
dupes = chars_with_sector[dupe_mask].sort_values(["gvkey", "eom"])

print(f"Total duplicated (gvkey, eom) pairs: {dupes[['gvkey','eom']].drop_duplicates().shape[0]}")
dupes.head(10)

In [ ]:
dupe_analyze = dupes.iloc[0:3]  # first 3 duplicate rows

# Pairwise comparisons (each is a boolean Series by column)
diff_01 = dupe_analyze.iloc[0] != dupe_analyze.iloc[1]
diff_02 = dupe_analyze.iloc[0] != dupe_analyze.iloc[2]
diff_12 = dupe_analyze.iloc[1] != dupe_analyze.iloc[2]

# Columns that differ in ANY of the three comparisons
differences = diff_01 | diff_02 | diff_12
different_columns = differences[differences].index.tolist()

print("Different columns across the first duplicated gvkey/eom rows:")
for col in different_columns:
    print(f"Column: {col}")
    print(f"Row 1: {dupe_analyze.iloc[0][col]}")
    print(f"Row 2: {dupe_analyze.iloc[1][col]}")
    print(f"Row 3: {dupe_analyze.iloc[2][col]}")
    print("-" * 40)

In [ ]:
#find dupes with gvkey 001045
dupes_001045 = dupes[dupes['gvkey'] == '001045']
# duplicates_001045 = dupes_001045[dupes_001045.duplicated(subset=['gvkey', 'eom'], keep=False)]
dupes_001045

In [ ]:
chars_with_sector.to_sql(
    "jkp_characteristics_with_sector",  # new table name
    engine,
    if_exists="replace",               # use "append" if you want to add to an existing table
    index=False,
    method="multi",
    chunksize=10_000,
)

In [ ]:
import multiprocessing as mp

download_workers = max(2, mp.cpu_count() - 2)
download_workers